# Postnatal Multiome analysis 

# Load packages

In [ ]:
library(Signac)
library(Seurat)
library(rtracklayer)
library(dplyr)
library(Matrix)
library(ggplot2)
library(cowplot)
require(biomaRt)
library(DoubletFinder)
library(RColorBrewer)
library(stringr)
set.seed(1234)

In [ ]:
my_c <- c('#fe864a', '#c9d179', '#95d0fc', '#fbdd7e', '#388004', '#c760ff', '#76cd26', '#ffbacd', '#d6b4fc',
        '#cfaf7b','#8b88f8', '#acc2d9', '#894585', '#82cbb2', '#d99b82', '#fdee73', '#5170d7', '#ffd8b1', 
        '#b7e1a1', '#82a67d', '#b7fffa', '#b790d4', '#f5674a', '#C79FEF', '#DAA520', '#A0522D')

# Data loading

 This genes are the ones used to perform the aligment on CellRanger using the costumized genome reference. It contains teh CreERT2 gene and e GFP gene.

In [ ]:
annotation <- import('genes2.gtf')

In [ ]:
annotation$gene_biotype[1:1780455] = annotation$gene_type[1:1780455]
seqlevelsStyle(annotation) <- "UCSC" #Support for other than Ensembl seqlevel style


In [ ]:
annotation$tx_id = annotation$transcript_id

In [ ]:
#load fragments data
P11_sample1_frags <- 'P11_sample1_atac_fragments.tsv.gz'

P11_sample2_frags <- 'P11_sample2_atac_fragments.tsv.gz'

P11_sample3_frags <- 'P11_sample3_atac_fragments.tsv.gz'

P60_sample1_frags <- 'P60_sample1_atac_fragments.tsv.gz'

P60_sample2_frags <- 'P60_sample2_atac_fragments.tsv.gz'

P60_TTR_sample3_frags <- 'P60_TTR_sample3_atac_fragments.tsv.gz'

In [ ]:
sample_names <- c('P11_sample1', 'P11_sample2', 'P11_sample3', 'P60_sample1', 'P60_sample2', 'P60_TTR_sample3')

In [ ]:
frags_path <- c(P11_sample1_frags, P11_sample2_frags, P11_sample3_frags, P60_sample1_frags, P60_sample2_frags, 
           P60_TTR_sample3_frags)

# Load all cell types data

In [ ]:
raw <- readRDS('Merged_multiome.rds')

In [ ]:
raw

In [ ]:
filtered <- readRDS('Integrated_multiome_clusters_joint_linked_cell_type.rds')

# Load subclustering data

In [ ]:
epi <- readRDS('Epithelial_integrated_noDBL_multiome_celltype_joint_linked.rds')

In [ ]:
endo <- readRDS('Endothelial_integrated_noDBL_multiome_celltype_joint_linked.rds')

In [ ]:
mesen <- readRDS('Mesenchymal_integrated_noDBL_multiome_celltype_joint_linked.rds')

In [ ]:
immune <- readRDS('Immune_integrated_multiome_celltype_joint_linked.rds')

In [ ]:
neugli <- readRDS('Neuronal_and_Glial_integrated_noDBL_multiome_celltype_joint_linked.rds')

In [ ]:
epi$Cell_type_group <- 'Epithelial cells'

In [ ]:
endo$Cell_type_group <- 'Endothelial cells'

In [ ]:
mesen$Cell_type_group <- 'Mesenchymal cells'

In [ ]:
immune$Cell_type_group <- 'Immune cells'

In [ ]:
neugli$Cell_type_group <- 'Neuronal and glial cells'

In [ ]:
subclus_samples <- c(epi, endo, mesen, immune, neugli)

In [ ]:
subclus_samples

In [ ]:
for(i in 1:length(subclus_samples)){
    DefaultAssay(subclus_samples[[i]]) <- 'RNA'
    subclus_samples[[i]] <- subclus_samples[[i]]
}

In [ ]:
subclus_samples

In [ ]:
merged_filt <-  merge(subclus_samples[[1]], y = subclus_samples[2:5])

In [ ]:
merged_filt

In [ ]:
merged_filt

In [ ]:
unique(merged_filt@meta.data$Cell_type)

In [ ]:
raw_merged <- subset(raw, cells = colnames(merged_filt))

In [ ]:
subcell.meta <- data.frame(CellID = colnames(merged_filt), Sub_Cell_type = merged_filt$Cell_type)

In [ ]:
new.metadata <- data.frame(CellID = colnames(merged_filt), Cell_type = merged_filt$Cell_type_group)

In [ ]:
raw_merged$Sub_Cell_type <- 'placeholder'

In [ ]:
raw_merged$Sub_Cell_type <- subcelltype_meta

In [ ]:
raw_merged$Cell_type <- new.metadata

In [ ]:
head(raw_merged@meta.data)

In [ ]:
merged <- raw_merged

In [ ]:
merged

In [ ]:
frags <- Fragments(merged)

In [ ]:
Fragments(merged) <- NULL

In [ ]:
for (i in 1:length(frags)){
    frags[[i]] <- UpdatePath(frags[[i]], new.path = frags_path[[i]])
}

In [ ]:
Fragments(merged) <- frags

In [ ]:
saveRDS(merged, 'noDBL_merged_raw.rds')

In [ ]:
merged <- readRDS('noDBL_merged_raw.rds')

In [ ]:
merged@assays

In [ ]:
unique(epi$Sample)

In [ ]:
unique(merged$Age)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
merged$nucleosome_group <- ifelse(merged$nucleosome_signal > 2.0, 'NS > 2.0', 'NS < 2.0')

frag_hist <- FragmentHistogram(object = merged, group.by = 'nucleosome_group', region = 'chr1-1-10000000')

frag_hist

ggsave(file="noDBL_noDBL_Merged_frag_hist.svg", plot=frag_hist, width=8, height=4)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 8, repr.plot.res = 300)

vln <- VlnPlot(merged, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_ATAC', 'nFeature_ATAC', 'nucleosome_signal', 'TSS.enrichment', 'pct_mito', 'pct_hb'), 
        ncol=4, pt.size=0, col = my_c) 

vln

ggsave(file="noDBL_noDBL_Merged_all_vln_prefilter.svg", plot=vln, width=15, height=8)


In [ ]:
options(repr.plot.width=30, repr.plot.height=5)
vlns_pre <- c()
for (sample in sample_names){
    s <- subset(merged, subset = Sample == sample)
    vln <- VlnPlot(object = s, features = c("nCount_RNA", "nFeature_RNA", "nCount_ATAC", "nFeature_ATAC", "TSS.enrichment", "nucleosome_signal", "pct_mito", "pct_hb"), 
                  ncol = 8, pt.size = 0)
    
    plot_name = paste("noDBL_noDBL_", sample, "_vln_prefilter.svg")
    #vlns_pre <- c(vlns_pre, vln)
    print(vln)
    
    ggsave(file=plot_name, plot=vln, width=30, height=5)
    
    
}

In [ ]:
DefaultAssay(merged) <- 'RNA' 

In [ ]:
vars_to_regress <- c('Sample')

In [ ]:
merged <- merged %>% 
                            NormalizeData(normalization.method = "LogNormalize", scale.factor = 10000) %>%
                            FindVariableFeatures(selection.method = "vst", nfeatures=4000, ) %>%
                            ScaleData() %>%
                            RunPCA() 

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
pca <- ElbowPlot(merged, ndims = 50)

pca

ggsave(file="noDBL_noDBL_merged_pca.svg", plot=pca, width=6, height=4)

In [ ]:
merged <- RunUMAP(merged, dims = 1:22, reduction = 'pca', n.neighbors = 15)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
dpl <- DimPlot(merged, reduction = 'umap', group.by = 'Sample', pt.size = 0.5, raster = TRUE, cols = my_c)

dpl

# Call peaks

In [ ]:
merged

In [ ]:
DefaultAssay(merged) <- 'ATAC'

In [ ]:
# call peaks using MACS2
peaks <- CallPeaks(object=merged,, 
                outdir='callpeaks_noDBL', cleanup=FALSE)

In [ ]:
# remove peaks on nonstandard chromosomes and in genomic blacklist regions
peaks <- keepStandardChromosomes(peaks, pruning.mode = "coarse")
peaks <- subsetByOverlaps(peaks, ranges = blacklist_mm10, invert = TRUE)

In [ ]:
macs2_counts <- FeatureMatrix(fragments = Fragments(merged), features = peaks, cells = colnames(merged))

In [ ]:
merged[['peaks']] <- CreateChromatinAssay(counts = macs2_counts, sep = c('-', '-'), fragments = Fragments(merged), annotation = annotation)

In [ ]:
merged@assays

In [ ]:
saveRDS(merged, 'noDBL_merged_peaks.rds')

# FRiP

In [ ]:
merged <- FRiP(merged, assay = 'peaks', total.fragments = "nCount_ATAC", col.name = 'FRiP', verbose = TRUE)

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 12, repr.plot.res = 300)

vln <- VlnPlot(merged, features=c('nCount_RNA', 'nFeature_RNA', 'nCount_peaks', 'nFeature_peaks', 'nucleosome_signal', 'TSS.enrichment', 'FRiP', 'pct_mito', 'pct_hb'), 
        ncol=3, pt.size=0, col = my_c) 

vln

ggsave(file="noDBL_merged_vln_pre_filter_frip.svg", plot=vln, width=15, height=12)


In [ ]:
saveRDS(merged, 'noDBL_merged_peaks.rds')

# Integration and clustering

In [ ]:
merged <- readRDS('noDBL_merged_peaks.rds')

## RNA

In [ ]:
table(merged$orig.ident)

In [ ]:
DefaultAssay(merged) <- 'RNA'

In [ ]:
merged <- merged %>% 
                            NormalizeData() %>%
                            FindVariableFeatures(selection.method = "vst", nfeatures=4000, ) %>%
                            ScaleData() %>%
                            RunPCA() 

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
pca <- ElbowPlot(merged, ndims = 50)

pca

ggsave(file="noDBL_Merged_pca.svg", plot=pca, width=6, height=4)

In [ ]:
merged <- RunUMAP(merged, dims = 1:22, reduction = 'pca', n.neighbors = 15, reduction.name="umapRNA")

In [ ]:
unique(merged$Age)

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
before_umap_s <- DimPlot(merged, reduction = 'umapRNA', group.by = 'Sample', pt.size = 0.5, raster = TRUE, cols = my_c)

before_umap_s

ggsave(file="noDBL_Merged_Sample_UMAP.svg", plot=before_umap_s, width=8, height=4)


options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
before_umap_a <- DimPlot(merged, reduction = 'umapRNA', group.by = 'Age', pt.size = 0.5, raster = TRUE, cols = my_c[2:3])

before_umap_a

ggsave(file="noDBL_Merged_Age_UMAP.svg", plot=before_umap_a, width=6, height=4)

options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
before_umap_a <- DimPlot(merged, reduction = 'umapRNA', group.by = 'Batch', pt.size = 0.5, raster = TRUE, cols = my_c[2:3])

before_umap_a

ggsave(file="noDBL_Merged_Batch_UMAP.svg", plot=before_umap_a, width=6, height=4)

In [ ]:
options(repr.plot.width=24, repr.plot.height=12)
dpl <- DimPlot(merged, group.by="orig.ident", split.by="orig.ident", reduction="umapRNA", shuffle=TRUE, cols=my_c, ncol=3, pt.size=0.5, label=FALSE)

dpl


ggsave(file="noDBL_Merged_each_sample_UMAP.svg", plot=dpl, width=24, height=12)

### Integration with Harmony

In [ ]:
library(harmony)

In [ ]:
integrated <- RunHarmony(merged, group.by.vars = 'Sample', reduction.save = 'harmonyRNA')

In [ ]:
integrated

In [ ]:
integrated <- RunUMAP(integrated, reduction = 'harmonyRNA', dims = 1:22, n.neighbors = 15, reduction.name="harmony_RNA")

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 4, repr.plot.res = 200)
after_umap_s <- DimPlot(integrated, reduction = 'harmony_RNA', group.by = 'Sample', pt.size = 0.5, raster = TRUE, cols = my_c)

after_umap_s

ggsave(file="noDBL_Harmony_RNA_sample_UMAP.svg", plot=after_umap_s, width=8, height=4)


options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
after_umap_a <- DimPlot(integrated, reduction = 'harmony_RNA', group.by = 'Age', pt.size = 0.2, raster = TRUE, cols = my_c[2:3])

after_umap_a

ggsave(file="noDBL_Harmony_RNA_Age_UMAP.svg", plot=after_umap_a, width=6, height=4)

options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
after_umap_b <- DimPlot(integrated, reduction = 'harmony_RNA', group.by = 'Batch', pt.size = 0.2, raster = TRUE, cols = my_c[2:3])

after_umap_b

ggsave(file="noDBL_Harmony_RNA_Batch_UMAP.svg", plot=after_umap_b, width=6, height=4)


In [ ]:
options(repr.plot.width=24, repr.plot.height=12)
dpl <- DimPlot(integrated, group.by="orig.ident", split.by="orig.ident", reduction="harmony_RNA", shuffle=TRUE, 
               cols=my_c, ncol=3, pt.size=0.2, label=FALSE)

dpl

ggsave(file="noDBL_Harmony_each_sample_UMAP.svg", plot=dpl, width=24, height=12)

In [ ]:
saveRDS(integrated, 'noDBL_merged_peaks_clusters.rds')

# Peaks

In [ ]:
integrated <- readRDS('noDBL_merged_peaks_clusters.rds')

## Normalization and dimensional reduction

In [ ]:
DefaultAssay(integrated)<-'peaks'
integrated<-FindTopFeatures(integrated,min.cutoff = NULL)
integrated<-RunTFIDF(integrated) #Run term frequency inverse document frequency (TF-IDF) normalization on a matrix.
integrated<-RunSVD(integrated)

In [ ]:
options(repr.plot.width = 5, repr.plot.height = 5, repr.plot.res = 200)
DepthCor(integrated,n=50,reduction = 'lsi')

In [ ]:
options(repr.plot.width=6, repr.plot.height=4)
ebl <- ElbowPlot(integrated, reduction = "lsi", n=50)

ebl

ggsave(file="noDBL_Merged_depthCorr_ebl.svg", plot=ebl, width=6, height=4)

In [ ]:
integrated <- RunUMAP(integrated,reduction = 'lsi',dims = 2:30,reduction.name = 'atac.umap')
integrated <- FindNeighbors(integrated,reduction = 'lsi',dims = 2:30)

In [ ]:
options(repr.plot.width=8, repr.plot.height=4)
dpl <- DimPlot(integrated, group.by="Sample", reduction="atac.umap", pt.size = 0.5, raster = TRUE, cols = my_c, label=FALSE)

dpl 

ggsave(file="noDBL_integrated_Peaks_Sample_UMAP.svg", plot=dpl, width=8, height=4)

## Integration with Harmony

In [ ]:
library(harmony)

In [ ]:
integrated <- RunHarmony(integrated, group.by.vars = 'Sample', reduction = 'lsi', assay.use = 'peaks', reduction.save = 'harmonyPeaks', project.dim = FALSE)

In [ ]:
# re-compute the UMAP using corrected LSI embeddings
integrated <- RunUMAP(integrated, dims = 2:30, reduction = 'harmonyPeaks', reduction.name = 'harmony_Peaks')

In [ ]:
options(repr.plot.width=8, repr.plot.height=4)
dpl <- DimPlot(integrated, group.by = 'Sample', pt.size = 0.5, raster=TRUE, cols=my_c, reduction = 'harmony_Peaks')

dpl

ggsave(file="noDBL_integrated_Peaks_Sample_UMAP.svg", plot=dpl, width=8, height=4)

In [ ]:
options(repr.plot.width=24, repr.plot.height=12)
dpl <- DimPlot(integrated, group.by="orig.ident", split.by="orig.ident", reduction="harmony_Peaks", shuffle=TRUE, cols=my_c, ncol=3, pt.size=0.5, raster = TRUE, label=FALSE)

dpl

ggsave(file="noDBL_integrated_Peaks_each_Sample_UMAP.svg", plot=dpl, width=24, height=12)

In [ ]:
options(repr.plot.width=8, repr.plot.height=4)
dpl <- DimPlot(integrated, group.by="Batch", reduction="harmony_Peaks", pt.size = 0.5, raster = TRUE, cols = my_c, label=FALSE)

dpl

ggsave(file="noDBL_integrated_Peaks_Batch_UMAP.svg", plot=dpl, width=8, height=4)

In [ ]:
options(repr.plot.width=16, repr.plot.height=6)
dpl <- DimPlot(integrated, group.by="orig.ident", split.by="Batch", reduction="harmony_Peaks", shuffle=TRUE, cols=my_c, ncol=5, pt.size=0.5, raster = TRUE, label=FALSE)

dpl

ggsave(file="noDBL_integrated_Peaks_each_batch_UMAP.svg", plot=dpl, width=16, height=6)

In [ ]:
saveRDS(integrated, 'noDBL_merged_peaks_clusters.rds')

# Joint vizualization

In [ ]:
integrated <- readRDS('noDBL_merged_peaks_clusters.rds')

In [ ]:
integrated@reductions

In [ ]:
integrated

In [ ]:
integrated <- FindMultiModalNeighbors(integrated, reduction.list = c('harmonyRNA', 'harmonyPeaks'),
                                       dims.list = list(1:22,2:30), verbose=TRUE)

In [ ]:
integrated <- RunUMAP(integrated, nn.name = "weighted.nn", assay = "RNA", reduction.name = 'multiome.umap',  verbose = TRUE)

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4, repr.plot.res = 200)
dpt <- DimPlot(integrated, group.by = "Sample", pt.size=0.5, raster = TRUE, cols = my_c, label = FALSE, repel = TRUE, reduction = 'multiome.umap')

dpt

ggsave(file="noDBL_integrated_clusters_Sample_Multiome.svg", plot=dpt, width=7, height=5)

In [ ]:
DefaultAssay(integrated) <- 'RNA'

In [ ]:
saveRDS(integrated, 'noDBL_merged_peaks_clusters_joint.rds')

# Cell types

In [ ]:
integrated <- readRDS('noDBL_merged_peaks_clusters_joint.rds')

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5, repr.plot.res = 200)
dmp <- DimPlot(object = integrated, label = FALSE,group.by='Cell_type', cols = my_c, 
               reduction = 'harmony_RNA', raster = TRUE, pt.size = 1.0)

dmp

ggsave(file="noDBL_Harmony_noDBL_Cell_type_RNA_umap.svg", plot=dmp, width=8, height=5)

In [ ]:
integrated <- SetIdent(integrated, value = 'Cell_type')

In [ ]:
integrated_predicted.markers <- FindAllMarkers(integrated, test.use="wilcox", only.pos=TRUE, max.cells.per.ident=15)

In [ ]:
write.csv(integrated_predicted.markers, 'noDBL_Markers_RNA_All.csv', row.names = TRUE)

In [ ]:
integrated_predicted.markers %>% group_by(cluster) %>%
                                slice_head(n = 10) %>%
                                ungroup() -> top10_predicted

In [ ]:
top10_predicted <- top10_predicted[!duplicated(top10_predicted$gene), ]

In [ ]:
options(repr.plot.width = 10, repr.plot.height = 4, repr.plot.res = 300)
dtp <- DotPlot(integrated, features = top10_predicted$gene, col.min = 0, col.max = 4, dot.scale = 4, scale=FALSE) + theme(axis.text.x=element_text(size=7, angle = 90),axis.text.y=element_text(size=7), 
                                                                                                 legend.position = 'bottom', legend.direction = 'horizontal', 
                                                                                                  legend.text = element_text(size=7)) + xlab('') + ylab('') & scale_color_distiller(palette = "YlGnBu", direction = 1)

dtp

ggsave(file="noDBL_All_integrated_dotplot_Cell_type.svg", plot=dtp, width=10, height=6)

# Marker genes on different umaps assays

## RNA

In [ ]:
DefaultAssay(integrated) <- 'RNA'

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5, repr.plot.res = 200)
dmp <- DimPlot(object = integrated, label = FALSE,group.by='Cell_type', cols = my_c, 
               reduction = 'harmony_RNA', raster = TRUE, pt.size = 1.0)

dmp

ggsave(file="noDBL_Harmony_noDBL_Cell_type_RNA_umap.svg", plot=dmp, width=8, height=5)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 25, repr.plot.res = 200)
ftp <- FeaturePlot(integrated, reduction = 'harmony_RNA', features = markers, 
             min.cutoff = 0, label.size = 2, ncol = 4, pt.size = 0.5, raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 


ggsave(file="noDBL_harmonyRNA_geneExpression.svg", plot=ftp, width=20, height=35)

## Peaks

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5, repr.plot.res = 200)
dmp <- DimPlot(object = integrated, label = FALSE,group.by='Cell_type', cols = my_c, 
               reduction = 'harmony_Peaks', raster = TRUE, pt.size = 1.0)

dmp

ggsave(file="noDBL_Harmony_noDBL_Cell_type_peaks_umap.svg", plot=dmp, width=8, height=5)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 25, repr.plot.res = 200)
ftp <- FeaturePlot(integrated, reduction = 'harmony_Peaks', features = markers, 
             min.cutoff = 0, label.size = 2, ncol = 4, pt.size = 0.5, raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 


ggsave(file="noDBL_harmonyPeaks_geneExpression.svg", plot=ftp, width=20, height=35)

## Joint

In [ ]:
options(repr.plot.width = 8, repr.plot.height = 5, repr.plot.res = 200)
dmp <- DimPlot(object = integrated, label = FALSE,group.by='Cell_type', cols = my_c, 
               reduction = 'multiome.umap', raster = TRUE, pt.size = 1.0)

dmp

ggsave(file="noDBL_Harmony_noDBL_Cell_type_joint_umap.svg", plot=dmp, width=8, height=5)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 25, repr.plot.res = 200)
ftp <- FeaturePlot(integrated, reduction = 'multiome.umap', features = markers, 
             min.cutoff = 0, label.size = 2, ncol = 4, pt.size = 0.5, raster=TRUE) & scale_color_distiller(palette = "YlGnBu", direction = 1)
ftp 


ggsave(file="noDBL_joint_geneExpression.svg", plot=ftp, width=20, height=35)

In [ ]:
integrated <- readRDS('noDBL_merged_peaks_clusters_joint.rds')

# Link peaks to genes

In [ ]:
integrated <- readRDS('noDBL_merged_peaks_clusters_joint.rds')

In [ ]:
library(BSgenome.Mmusculus.UCSC.mm10)

In [ ]:
DefaultAssay(integrated) <- 'peaks'

In [ ]:
# first compute the GC content for each peak
integrated <- RegionStats(integrated, genome = BSgenome.Mmusculus.UCSC.mm10)

In [ ]:
integrated[['ATAC']] <- NULL

In [ ]:
integrated

In [ ]:
integrated <- LinkPeaks(integrated, peak.assay = "peaks", expression.assay = "RNA")

In [ ]:
integrated

In [ ]:
saveRDS(integrated, 'noDBL_merged_peaks_clusters_joint_linked.rds')

# Peaks for marker genes

In [ ]:
integrated <- readRDS('noDBL_merged_peaks_clusters_joint_linked.rds')

In [ ]:
integrated

In [ ]:
DefaultAssay(integrated) <- 'peaks'

In [ ]:
Annotation(integrated) <- annotation

In [ ]:
integrated$splitby <- paste0(integrated$Age, "-", integrated$Cell_type)

In [ ]:
markers <- c('Folr1', 'Clic6', 'Krt18', 'Ttr', 
             'Dcn', 'Lama2', 'Adam12', 'Cfh', 
             'Plvap', 'Pecam1', 'Flt1', 'Plcb1', 
             'Cadm1', 'Dscam', 'Mbp','Snap25',
             'Cd83', 'Ptprc', 'C1qa', 'Txk')

In [ ]:
epi <- c('Folr1', 'Clic6', 'Krt18', 'Ttr')

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 5, repr.plot.res = 200)

In [ ]:
for (gene in epi){
    cvp <- CoveragePlot(integrated, region = gene, features = gene, group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("noDBL_", gene, "_igv_tracks_epi.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)
    
}

In [ ]:
mesen <- c('Dcn', 'Lama2', 'Adam12', 'Cfh')

In [ ]:
for (gene in mesen){
    cvp <- CoveragePlot(integrated, region = gene, features = gene, group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("noDBL_", gene, "_igv_tracks_mesen.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)
    
}

In [ ]:
endo <- c('Plvap', 'Pecam1', 'Flt1', 'Plcb1')

In [ ]:
for (gene in endo){
    cvp <- CoveragePlot(integrated, region = gene, features = gene, group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("noDBL_", gene, "_igv_tracks_endo.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)
    
}

In [ ]:
neugli <- c('Cadm1', 'Dscam', 'Mbp','Snap25')

In [ ]:
for (gene in neugli){
    cvp <- CoveragePlot(integrated, region = gene, features = gene, group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("noDBL_", gene, "_igv_tracks_neugli.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)
    
}

In [ ]:
immune <- c('Cd83', 'Ptprc', 'C1qa', 'Cd86', 'Cd74')

In [ ]:
for (gene in immune){
    cvp <- CoveragePlot(integrated, region = gene, features = gene, group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("noDBL_", gene, "_igv_tracks_immune.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)
    
}

# Call peaks on clusters

In [ ]:
integrated <- readRDS('noDBL_merged_peaks_clusters_joint_linked.rds')

In [ ]:
integrated

In [ ]:
DefaultAssay(integrated) <- 'peaks'

In [ ]:
# call peaks using MACS2
peaks <- CallPeaks(object=integrated, group.by='Cell_type', 
                outdir='callpeaks_noDBL_ct', cleanup=FALSE)

In [ ]:
peaks_df <- as.data.frame(peaks)

In [ ]:
head(peaks_df)

In [ ]:
write.csv(peaks_df, file = "peaks.csv", row.names = FALSE)

In [ ]:
write.table(peaks_df, file = "peaks.tsv", sep = "\t", row.names = FALSE, quote = FALSE)

# Find DA peaks 

In [ ]:
integrated <- readRDS('noDBL_merged_peaks_clusters_joint_linked.rds')

## Load HOMER annotation

In [ ]:
homer_anno <- read.delim('hommer_anno_all.txt')

In [ ]:
homer_anno$Peaks <- paste0(homer_anno$Chr, '-', homer_anno$Start, "-", homer_anno$End)

In [ ]:
homer_anno$Annotation <- sapply(strsplit(homer_anno$Annotation, " "), `[`, 1)

**The cell below creats the function to find the overlaps between da peaks and the ones annotated using homer and then plots them on a donut plot**

In [ ]:
library(GenomicRanges)
library(ggplot2)

create_peak_overlap_plot <- function(df_peaks, homer_anno, my_c, plot_name) {
  
  # Create GRanges objects
  marker_gr <- GRanges(seqnames = sub("-.*", "", df_peaks$gene),
                       ranges = IRanges(start = as.integer(sub(".*-(\\d+)-.*", "\\1", df_peaks$gene)),
                                        end = as.integer(sub(".*-", "", df_peaks$gene))))
  
  peak_gr <- GRanges(seqnames = sub("-.*", "", homer_anno$Peaks),
                     ranges = IRanges(start = as.integer(sub(".*-(\\d+)-.*", "\\1", homer_anno$Peaks)),
                                      end = as.integer(sub(".*-", "", homer_anno$Peaks))))
  
  # Find overlaps
  overlaps <- findOverlaps(marker_gr, peak_gr)
  epi_valid_markers <- homer_anno[queryHits(overlaps), ]
  
  # Get peaks info
  peaks_info <- table(unlist(epi_valid_markers$Annotation))
  df_info <- data.frame(peaks_info)
  colnames(df_info) <- c('Type', 'Frequency')
  
  # Compute percentages and cumulative sums
  df_info$fraction <- df_info$Frequency / sum(df_info$Frequency)
  df_info$ymax <- cumsum(df_info$fraction)
  df_info$ymin <- c(0, head(df_info$ymax, n=-1))
  df_info$labelPosition <- (df_info$ymax + df_info$ymin) / 2
  df_info$label <- paste0(df_info$Type, "\n", df_info$Frequency)
  
  # Create the plot
  p <- ggplot(df_info, aes(ymax=ymax, ymin=ymin, xmax=3, xmin=2, fill=Type)) +
    geom_rect() +
    scale_fill_manual(values=my_c[9:24]) +
    geom_text(x=3.8, aes(y=labelPosition, label=label, color=Type), size=6, angle=45) +
    scale_colour_manual(values=my_c[9:24]) +
    coord_polar(theta="y") +
    xlim(c(-1, 4)) +
    theme_void() +
    theme(legend.position = "right")

  ggsave(file=plot_name, plot=p)
  
  return(p)
}


## Cell types

In [ ]:
da_peaks <- FindAllMarkers(
  object = integrated,
  test.use = 'LR',
  min.pct = 0.1)

In [ ]:
da_peaks <- da_peaks[da_peaks$p_val_adj < 0.05, ]

In [ ]:
write.csv(da_peaks, 'DA_peaks_all.csv')

In [ ]:
da_peaks <- read.csv('DA_peaks_all.csv', row.names = 1)

In [ ]:
head(da_peaks)

### Epithelial cells

In [ ]:
epi_peaks <- subset(da_peaks, cluster == 'Epithelial cells')

In [ ]:
dim(epi_peaks)

In [ ]:
options(repr.plot.width = 20, repr.plot.height = 10, repr.plot.res = 300)

In [ ]:
plot <- create_peak_overlap_plot(epi_peaks, homer_anno, my_c, plot_name='Donut_epi.svg')

plot

### Endothelial cells

In [ ]:
integrated_Peaks <- subset(da_peaks, cluster == 'Endothelial cells')

In [ ]:
dim(integrated_Peaks)

#### Donut plots por types of peaks

In [ ]:
plot <- create_peak_overlap_plot(integrated_Peaks, homer_anno, my_c, plot_name='Donut_endo.svg')

plot

### Mesenchymal cells

In [ ]:
mesen_peaks <- subset(da_peaks, cluster == 'Mesenchymal cells')

In [ ]:
dim(mesen_peaks)

In [ ]:
dim(mesen_peaks)

#### Donut plots por types of peaks

In [ ]:
plot <- create_peak_overlap_plot(mesen_peaks, homer_anno, my_c, plot_name='Donut_mesen.svg')

plot

### Immune cells

In [ ]:
immune_peaks <- subset(da_peaks, cluster == 'Immune cells')

In [ ]:
dim(immune_peaks)

In [ ]:
dim(immune_peaks)

#### Donut plots por types of peaks

In [ ]:
plot <- create_peak_overlap_plot(immune_peaks, homer_anno, my_c, plot_name='Donut_immune.svg')

plot

### Neuronal and Glial cells

In [ ]:
neugli_peaks <- subset(da_peaks, cluster == 'Neuronal and glial cells')

In [ ]:
dim(neugli_peaks)

In [ ]:
dim(neugli_peaks)

#### Donut plots por types of peaks

In [ ]:
plot <- create_peak_overlap_plot(neugli_peaks, homer_anno, my_c, plot_name='Donut_neugli.svg')

plot

## Condition

In [ ]:
Idents(integrated) <- "Age"

In [ ]:
da_peaks_age <- FindAllMarkers(
  object = integrated,
  assay = 'peaks',
  test.use = 'wilcox',
  min.pct = 0.01
)

In [ ]:
dim(da_peaks_age)

In [ ]:
da_peaks_age <- da_peaks_age[da_peaks_age$p_val_adj < 0.05, ]

In [ ]:
write.csv(da_peaks_age, 'DA_peaks_all_age.csv')

In [ ]:
dim(da_peaks_age)

In [ ]:
p11_peaks <- subset(da_peaks_age, cluster == 'P11')

In [ ]:
dim(p11_peaks)

In [ ]:
p60_peaks <- subset(da_peaks_age, cluster == 'P60')

In [ ]:
dim(p60_peaks)

# Plots for figures

In [ ]:
integrated$log10_ngenes <- log10(integrated$nFeature_RNA)

In [ ]:
Idents(integrated) <- "Cell_type"

In [ ]:
options(repr.plot.width = 25, repr.plot.height = 9, repr.plot.res = 300)

vln <- VlnPlot(integrated, features=c('log10_ngenes'), group.by = 'Cell_type',
        ncol=4, pt.size=0, col = my_c)  + scale_y_continuous(limits = c(0,5))

vln

ggsave(file="noDBL_noDBL_integrated_all_vln_log10genes_Cell_Types.svg", plot=vln, width=25, height=9)


In [ ]:
options(repr.plot.width = 20, repr.plot.height = 7, repr.plot.res = 300)

vln <- VlnPlot(integrated, features=c('pct_mito'), group.by = 'Cell_type',
        ncol=4, pt.size=0, col = my_c) + scale_y_continuous(limits = c(0,20))

vln

ggsave(file="noDBL_noDBL_integrated_all_vln_pctmito_Cell_Types.svg", plot=vln, width=20, height=7)


In [ ]:
options(repr.plot.width = 25, repr.plot.height = 9, repr.plot.res = 300)

vln <- VlnPlot(integrated, features=c('log10_ngenes'), group.by = 'Sample',
        ncol=4, pt.size=0, col = my_c)  + scale_y_continuous(limits = c(0,5))

vln

ggsave(file="noDBL_noDBL_integrated_all_vln_log10genes_Sample.svg", plot=vln, width=25, height=9)


In [ ]:
options(repr.plot.width = 20, repr.plot.height = 7, repr.plot.res = 300)

vln <- VlnPlot(integrated, features=c('pct_mito'), group.by = 'Sample',
        ncol=4, pt.size=0, col = my_c) + scale_y_continuous(limits = c(0,20))

vln

ggsave(file="noDBL_noDBL_integrated_all_vln_pctmito_Sample.svg", plot=vln, width=20, height=7)


In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated, label = FALSE,group.by='Age', cols = my_c[2:3], 
               reduction = 'harmony_RNA', raster = TRUE, pt.size = 1)

dmp

ggsave(file="noDBL_Harmony_noDBL_Age_RNA_umap.svg", plot=dmp, width=6, height=4)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated, label = FALSE,group.by='Age', cols = my_c[2:3], 
               reduction = 'harmony_Peaks', raster = TRUE, pt.size = 1)

dmp

ggsave(file="noDBL_Harmony_noDBL_Age_peaks_umap.svg", plot=dmp, width=6, height=4)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated, label = FALSE,group.by='Age', cols = my_c[2:3], 
               reduction = 'multiome.umap', raster = TRUE, pt.size = 1)

dmp

ggsave(file="noDBL_Harmony_noDBL_Age_joint_umap.svg", plot=dmp, width=6, height=4)

In [ ]:
DefaultAssay(integrated) <- 'RNA'

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5, repr.plot.res = 300)

vln <- VlnPlot(integrated, features=c('Ttr', 'eGFP', 'CreERT2'), group.by = 'Sample',
        ncol=4, pt.size=0, col = my_c) 

vln

ggsave(file="noDBL_vln_TTR_eGFP_CreERT2_all_samples.svg", plot=vln, width=14, height=5)

In [ ]:
TTR_mouse <- subset(integrated, subset = Sample == 'P60_TTR_sample3')

In [ ]:
TTR_mouse

In [ ]:
options(repr.plot.width = 14, repr.plot.height = 5, repr.plot.res = 300)

vln <- VlnPlot(TTR_mouse, features=c('Ttr', 'eGFP', 'CreERT2'), group.by = c('Cell_type'),
        ncol=4, pt.size=0, col = my_c) 

vln
ggsave(file="noDBL_vln_TTR_eGFP_CreERT2_TTR_mouse.svg", plot=vln, width=14, height=5)

## IGV tracks

In [ ]:
integrated <- readRDS('noDBL_merged_peaks_clusters_joint_linked_motif.rds')

In [ ]:
integrated

In [ ]:
DefaultAssay(integrated) <- 'peaks'

In [ ]:
Annotation(integrated) <- annotation

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 10, repr.plot.res = 200)

#### Ttr

In [ ]:
regions <- c('chr18-20674033-20675277', 'chr18-20662176-20663114')

In [ ]:
ranges.show <- StringToGRanges(regions)

cvp <- CoveragePlot(integrated, region = 'Ttr', feature = 'Ttr', region.highlight = ranges.show, group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 8000, 
                        extend.downstream = 2000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("highlight_noDBL_", 'Ttr', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

In [ ]:
ranges.show <- StringToGRanges(regions)

cvp <- CoveragePlot(integrated, region = 'Ttr', feature = 'Ttr', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 8000, 
                        extend.downstream = 2000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("no_highlight_noDBL_", 'Ttr', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

#### Pcam1

In [ ]:
regions <- c('chr11-106674418-106675320')

In [ ]:
ranges.show <- StringToGRanges(regions)

cvp <- CoveragePlot(integrated,
                        region = 'Pecam1', features = 'Pecam1', region.highlight = ranges.show, group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 2000, 
                        extend.downstream = 2000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("highlight_noDBL_", 'Pcam1', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

In [ ]:
cvp <- CoveragePlot(integrated,
                        region = 'Pecam1', features = 'Pecam1', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 2000, 
                        extend.downstream = 2000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("no_highlight_noDBL_", 'Pcam1', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

#### Clic6

In [ ]:
regions <- c('chr16-92510704-92511998', 'chr16-92500163-92501512')

In [ ]:
ranges.show <- StringToGRanges(regions)

cvp <- CoveragePlot(integrated,
                        region = 'Clic6', features = 'Clic6', region.highlight = ranges.show, group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 2000, 
                        extend.downstream = 11000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("highlight_noDBL_", 'Clic6', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

In [ ]:


cvp <- CoveragePlot(integrated,
                        region = 'Clic6', features = 'Clic6', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 2000, 
                        extend.downstream = 11000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("no_highlight_noDBL_", 'Clic6', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

#### Ptprc

In [ ]:
cvp <- CoveragePlot(integrated,
                        region = 'Ptprc', features = 'Ptprc', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 2000, 
                        extend.downstream = 2000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("highlight_noDBL_", 'Ptprc', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

In [ ]:
cvp <- CoveragePlot(integrated,
                        region = 'Ptprc', features = 'Ptprc', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 2000, 
                        extend.downstream = 2000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("no_highlight_noDBL_", 'Ptprc', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

#### Dcn

In [ ]:
#regions <- c()

In [ ]:
#ranges.show <- StringToGRanges(regions)

cvp <- CoveragePlot(integrated,
                        region = 'Dcn', features = 'Dcn', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 2000, 
                        extend.downstream = 2000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("highlight_noDBL_", 'Dcn', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

In [ ]:
cvp <- CoveragePlot(integrated,
                        region = 'Dcn', features = 'Dcn', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 2000, 
                        extend.downstream = 2000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("no_highlight_noDBL_", 'Dcn', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

#### Ccl2

In [ ]:
# regions <- c()

In [ ]:
# ranges.show <- StringToGRanges(regions)

cvp <- CoveragePlot(integrated,
                        region = 'Ccl2', features = 'Ccl2', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 20000, 
                        extend.downstream = 2000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("highlight_noDBL_", 'Ccl2', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

In [ ]:
cvp <- CoveragePlot(integrated,
                        region = 'Ccl2', features = 'Ccl2', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 20000, 
                        extend.downstream = 2000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("no_highlight_noDBL_", 'Ccl2', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

#### Csf1

In [ ]:
regions <- c('chr3-107759465-107760957')

In [ ]:
ranges.show <- StringToGRanges(regions)

cvp <- CoveragePlot(integrated,
                        region = 'Csf1', features = 'Csf1', region.highlight = ranges.show, group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 2000, 
                        extend.downstream = 2000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("highlight_noDBL_", 'Csf1', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

In [ ]:
cvp <- CoveragePlot(integrated,
                        region = 'Csf1', features = 'Csf1', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 20000, 
                        extend.downstream = 20000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("no_highlight_noDBL_", 'Csf1', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

#### Cxcl1

In [ ]:
# regions <- c()

In [ ]:
# ranges.show <- StringToGRanges(regions)

cvp <- CoveragePlot(integrated,
                        region = 'Cxcl1', features = 'Cxcl1', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("highlight_noDBL_", 'Cxcl1', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

In [ ]:
cvp <- CoveragePlot(integrated,
                        region = 'Cxcl1', features = 'Cxcl1', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("no_highlight_noDBL_", 'Cxcl1', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

#### Cebpd

In [ ]:
regions <- c('chr16-15886577-15888202')

In [ ]:
ranges.show <- StringToGRanges(regions)

cvp <- CoveragePlot(integrated,
                        region = 'Cebpd', features = 'Cebpd', region.highlight = ranges.show, group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("highlight_noDBL_", 'Cebpd', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

In [ ]:
cvp <- CoveragePlot(integrated,
                        region = 'Cebpd', features = 'Cebpd', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("no_highlight_noDBL_", 'Cebpd', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

#### Il6

In [ ]:
# regions <- c()

In [ ]:
# ranges.show <- StringToGRanges(regions)

cvp <- CoveragePlot(integrated,
                        region = 'Il6', features = 'Il6', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("highlight_noDBL_", 'Il6', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

In [ ]:
cvp <- CoveragePlot(integrated,
                        region = 'Il6', features = 'Il6', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("no_highlight_noDBL_", 'Il6', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

#### Steap4

In [ ]:
# regions <- c()

In [ ]:
# ranges.show <- StringToGRanges(regions)

cvp <- CoveragePlot(integrated,
                        region = 'Steap4', features = 'Steap4', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("highlight_noDBL_", 'Steap4', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

In [ ]:
cvp <- CoveragePlot(integrated,
                        region = 'Steap4', features = 'Steap4', group.by = 'Cell_type',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = my_c)
    
    print(cvp)
    
    plot_name = paste0("no_highlight_noDBL_", 'Steap4', "_igv_tracks.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

## IGV tracks splited by age

In [ ]:
options(repr.plot.width = 15, repr.plot.height = 5, repr.plot.res = 200)

### Epithelial cells

#### H2-Eb1

In [ ]:
regions <- c('chr17-34309331-34310181', 'chr17-34305625-34306003', 'chr17-34313278-34313756')

In [ ]:
ranges.show <- StringToGRanges(regions)

cvp <- CoveragePlot(integrated,
                        region = 'H2-Eb1', features = 'H2-Eb1', region.highlight = ranges.show, idents = c('Epithelial cells'),
                    split.by = 'Age',
                    group.by = 'Cell_type', annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = c(my_c[2], my_c[3]))
    
    print(cvp)
    
    plot_name = paste0("Epi_Age_highlight_noDBL_", 'H2-Eb1', "_igv_tracks_all_cells.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

#### H2-Aa

In [ ]:
regions <- c('chr17-34287543-34288079')

In [ ]:
ranges.show <- StringToGRanges(regions)

cvp <- CoveragePlot(integrated,
                        region = 'H2-Aa', features = 'H2-Aa', region.highlight = ranges.show, idents = c('Epithelial cells'),
                    split.by = 'Age',
                    group.by = 'Cell_type', annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = c(my_c[2], my_c[3]))
    
    print(cvp)
    
    plot_name = paste0("Epi_Age_highlight_noDBL_", 'H2-Aa', "_igv_tracks_all_cells.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

### Endothelial cells

#### H2-K1

In [ ]:
cvp <- CoveragePlot(integrated,
                        region = 'H2-K1', features = 'H2-K1', idents = c('Endothelial cells'),
                    group.by = 'Cell_type', annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                    split.by = 'Age',
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = c(my_c[2], my_c[3]))
    
    print(cvp)
    
    plot_name = paste0("Endo_Age_no_highlight_noDBL_", 'H2-K1', "_igv_tracks_all_cells.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

#### H2-D1

In [ ]:
cvp <- CoveragePlot(integrated,
                        region = 'H2-D1', features = 'H2-D1', idents = c('Endothelial cells'), split.by = 'Age',
                    group.by = 'Cell_type', annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = c(my_c[2], my_c[3]))
    
    print(cvp)
    
    plot_name = paste0("Endo_Age_no_highlight_noDBL_", 'H2-D1', "_igv_tracks_all_cells.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

### Mesenchymal cells

#### H2-K1

In [ ]:
cvp <- CoveragePlot(integrated,
                        region = 'H2-K1', features = 'H2-K1', idents = c('Mesenchymal cells'), split.by = 'Age',
                    group.by = 'Cell_type', annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = c(my_c[2], my_c[3]))
    
    print(cvp)
    
    plot_name = paste0("Mesen_Age_no_highlight_noDBL_", 'H2-K1', "_igv_tracks_all_cells.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

#### H2-D1

In [ ]:
cvp <- CoveragePlot(integrated,
                        region = 'H2-D1', features = 'H2-D1', idents = c('Mesenchymal cells'), split.by = 'Age',
                    group.by = 'Cell_type', annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = c(my_c[2], my_c[3]))
    
    print(cvp)
    
    plot_name = paste0("Mesen_Age_no_highlight_noDBL_", 'H2-D1', "_igv_tracks_all_cells.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

### Immune cells

#### H2-Eb1

In [ ]:
cvp <- CoveragePlot(integrated,
                        region = 'H2-Eb1', features = 'H2-Eb1', idents = c('Immune cells'), group.by = 'Cell_type',
                    split.by = 'Age',
                        annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = c(my_c[2], my_c[3]))
    
    print(cvp)
    
    plot_name = paste0("Immune_Age_no_highlight_noDBL_", 'H2-Eb1', "_igv_tracks_all_cells.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

#### H2-Aa

In [ ]:
ranges.show <- StringToGRanges(regions)

cvp <- CoveragePlot(integrated,
                        region = 'H2-Aa', features = 'H2-Aa', idents = c('Immune cells'), group.by = 'Cell_type', 
                    split.by = 'Age',
                    annotation = TRUE, peaks = TRUE, links = TRUE, extend.upstream = 1000, 
                        extend.downstream = 1000, ncol=1) & scale_fill_manual(values = c(my_c[2], my_c[3]))
    
    print(cvp)
    
    plot_name = paste0("Immune_Age_no_highlight_noDBL_", 'H2-Aa', "_igv_tracks_all_cells.svg")
    
    ggsave(file=plot_name, plot=cvp, width=15, height=5)

# Xist counts to try to separate male and female cells 

In [ ]:
integrated <- readRDS('noDBL_merged_peaks_clusters_joint_linked_motif.rds')

In [ ]:
DefaultAssay(integrated) <- 'RNA'

In [ ]:
integrated

In [ ]:
xist_counts <- GetAssayData(integrated, assay = "RNA", layer = "counts")[ "Xist",]

In [ ]:
integrated$'Xist-counts' <- xist_counts

In [ ]:
integrated$'Female' <- integrated$`Xist-counts` > 0

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated, label = FALSE,group.by='Female', cols = my_c[2:3], 
               reduction = 'harmony_RNA', raster = TRUE, pt.size = 1)

dmp

ggsave(file="Plots/Gender_inference_Harmony_noDBL_RNA_umap_all_cells.svg", plot=dmp, width=6, height=4)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated, label = FALSE,group.by='Female', cols = my_c[2:3], 
               reduction = 'harmony_Peaks', raster = TRUE, pt.size = 1)

dmp

ggsave(file="Plots/Gender_inference_Harmony_noDBL_peaks_umap_all_cells.svg", plot=dmp, width=6, height=4)

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4, repr.plot.res = 200)
dmp <- DimPlot(object = integrated, label = FALSE,group.by='Female', cols = my_c[2:3], 
               reduction = 'multiome.umap', raster = TRUE, pt.size = 1)

dmp

ggsave(file="Plots/Gender_inference_Harmony_noDBL_joint_umap_all_cells.svg", plot=dmp, width=6, height=4)

# Session info

In [ ]:
print(sessioninfo::session_info())